# Single Prompt vs 4-Step Pipeline: unified evaluation comparison

**Purpose:** Unify the evaluation functions of the existing 4-step pipeline and single-prompt for a fair comparison

**Identified issue:**
- Existing 4-step: **micro F1** (per-cell correctness) + scope filtering
- Single prompt: **macro F1** (mean F1 across the 4 categories) + no filtering
- -> even identical predictions can yield very different scores

**This notebook:** Recompute by applying the same evaluation functions to both methods

In [1]:
import os
import json
import csv
import numpy as np
import pandas as pd
from collections import Counter
from IPython.display import display, HTML

## Configuration

In [2]:
DATA_DIR = "data"
GOLD_DIR = os.path.join(DATA_DIR, "gold")

# 4-step pipeline results directory
FOURSTEP_DIRS = {
    "GPT-5": "results/gpt5/raw",
    "GPT-4o": "results/gpt4o/raw",
}

# Single-prompt results directory
SINGLE_DIRS = {
    "GPT-5": "appendix/results/single_prompt/gpt5/raw",
    "GPT-4o": "appendix/results/single_prompt/gpt4o/raw",
}

TECHNIQUES = ["CoT"]  # Single prompt compares CoT only
ITERATIONS = range(1, 6)

print("✅ Configuration complete")

✅ 설정 완료


## Unified evaluation functions

Uses the evaluation method of the paper (Eq. 1~6) and the existing 4-step pipeline as-is:
- **Step 3 (Eq. 4):** scope filtering -> F1_PerceptionTable (triplet micro F1)
- **Step 4 (Eq. 5~6):** scope filtering -> Positive-F1_InterpretationTable
  - After **integrating (union)** preferences ∪ constraints, mean F1 per positive cell
  - Same as `integrate_tables()` + `evaluate_tables()` in `10_evaluation_step4_gpt5.ipynb`

In [3]:
def normalize_name(x):
    return x.strip().lower()


def f1_score(tp, fp, fn):
    if tp == 0:
        return 0.0
    prec = tp / (tp + fp) if tp + fp > 0 else 0.0
    rec = tp / (tp + fn) if tp + fn > 0 else 0.0
    return 2 * prec * rec / (prec + rec) if prec + rec > 0 else 0.0


# ── Step 3: paper Eq.4 / same as the existing pipeline's score_table() ──
# F1_PerceptionTable = micro F1 over triplets (p, r, perception) + scope filter
def eval_step3_micro(gold_table, pred_table, scope_p, scope_r):
    """Paper Eq.1,4 — F1_PerceptionTable (scope filtering -> micro F1)."""
    scope_r_lower = {normalize_name(r) for r in scope_r}
    
    gold_filtered = [g for g in gold_table
                     if g['participant'] in scope_p
                     and normalize_name(g['restaurant']) in scope_r_lower]
    pred_filtered = [p for p in pred_table
                     if p['participant'] in scope_p
                     and normalize_name(p['restaurant']) in scope_r_lower]
    
    gold_map = {(g['participant'], normalize_name(g['restaurant'])):
                g.get('perception', g.get('sentiment', '')).strip()
                for g in gold_filtered}
    pred_map = {(p['participant'], normalize_name(p['restaurant'])):
                p.get('sentiment', p.get('perception', '')).strip()
                for p in pred_filtered}
    
    tp = fp = fn = 0
    for key, gv in gold_map.items():
        if key in pred_map:
            if gv == pred_map[key]:
                tp += 1
            else:
                fn += 1
        else:
            fn += 1
    for key in pred_map:
        if key not in gold_map:
            fp += 1
    return {'F1_perception_micro': f1_score(tp, fp, fn)}


# ── Step 3: macro F1 (reference only — not in the paper) ──
def eval_step3_macro(gold_table, pred_table):
    """Reference macro F1 (mean of F1 across the 4 categories)."""
    gold_lookup = {(i['participant'], i['restaurant']):
                   i.get('perception', i.get('sentiment', 'Neutral'))
                   for i in gold_table}
    pred_lookup = {(i['participant'], i['restaurant']):
                   i.get('sentiment', i.get('perception', 'Neutral'))
                   for i in pred_table}
    all_keys = set(gold_lookup) | set(pred_lookup)
    f1_cats = []
    for cat in ['Positive', 'Negative', 'Neutral', 'Mix']:
        tp = fp = fn = 0
        for key in all_keys:
            gv = gold_lookup.get(key, 'Neutral')
            pv = pred_lookup.get(key, 'Neutral')
            if gv == cat and pv == cat: tp += 1
            elif gv != cat and pv == cat: fp += 1
            elif gv == cat and pv != cat: fn += 1
        f1_cats.append(f1_score(tp, fp, fn))
    return {'F1_perception_macro': np.mean(f1_cats)}


# ── Step 4: paper Eq.5~6 / same as 10_evaluation_step4_gpt5.ipynb in the existing pipeline ──

def parse_factors(s):
    """factor string -> set. 'A1,A2' -> {'A1','A2'}, 'None' -> set()."""
    if not s or str(s).strip().lower() == 'none':
        return set()
    return {f.strip() for f in str(s).split(',') if f.strip() and f.strip().lower() != 'none'}


def positive_f1(gold_set, pred_set):
    """Paper Eq.6 — per-cell factor-set F1."""
    inter = len(gold_set & pred_set)
    prec = inter / len(pred_set) if pred_set else 0.0
    rec = inter / len(gold_set) if gold_set else 0.0
    return 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0


def integrate_tables(pref_tbl, cons_tbl, pref_col='preferences', cons_col='constraints'):
    """
    Build the paper's Interpretation Table:
    Union of preferences ∪ constraints per (participant, restaurant).
    Same as integrate_tables() in 10_evaluation_step4_gpt5.ipynb.
    """
    union_map = {}

    for row in pref_tbl:
        key = (row['participant'], normalize_name(row['restaurant']))
        union_map.setdefault(key, set()).update(parse_factors(row.get(pref_col, 'None')))

    for row in cons_tbl:
        key = (row['participant'], normalize_name(row['restaurant']))
        union_map.setdefault(key, set()).update(parse_factors(row.get(cons_col, 'None')))

    return [
        {'participant': p, 'restaurant': r,
         'integrated': ','.join(sorted(factors)) if factors else 'None'}
        for (p, r), factors in union_map.items()
    ]


def evaluate_tables(gold_table, pred_table, factor_col):
    """
    Paper Eq.5 — Positive-F1: mean per-cell F1 over the D_positive cells
    + None Accuracy: fraction of gold=None cells where pred is also None
    Same as evaluate_tables() in 10_evaluation_step4_gpt5.ipynb.
    """
    gold = {(it['participant'], normalize_name(it['restaurant'])):
            parse_factors(it.get(factor_col, 'None')) for it in gold_table}
    pred = {(it['participant'], normalize_name(it['restaurant'])):
            parse_factors(it.get(factor_col, 'None')) for it in pred_table}

    pos_sum = pos_cnt = 0
    none_sum = none_cnt = 0

    for key, gold_set in gold.items():
        pred_set = pred.get(key, set())
        if gold_set:                                   # positive cell
            pos_sum += positive_f1(gold_set, pred_set)
            pos_cnt += 1
        else:                                          # gold == None cell
            none_sum += 1.0 if not pred_set else 0.0
            none_cnt += 1

    pos_mean  = pos_sum  / pos_cnt  if pos_cnt  else 0.0
    none_mean = none_sum / none_cnt if none_cnt else 1.0
    return pos_mean, none_mean


def eval_step4_paper(gold_pref, gold_cons, pred_pref, pred_cons, scope_p, scope_r):
    """
    Paper's Step 4 evaluation: Positive-F1_InterpretationTable
    1) scope filtering
    2) preferences ∪ constraints -> Interpretation Table (integrated)
    3) Compute Positive-F1 + None Accuracy on the integrated table
    + Also returns the individual (pref, cons) results (for reference)
    """
    scope_r_lower = {normalize_name(r) for r in scope_r}
    
    def filter_table(table):
        return [t for t in table
                if t['participant'] in scope_p
                and normalize_name(t['restaurant']) in scope_r_lower]
    
    gp, gc = filter_table(gold_pref), filter_table(gold_cons)
    pp, pc = filter_table(pred_pref), filter_table(pred_cons)
    
    # Individual evaluation (for reference)
    pref_pos_f1, pref_none_acc = evaluate_tables(gp, pp, 'preferences')
    cons_pos_f1, cons_none_acc = evaluate_tables(gc, pc, 'constraints')
    
    # ★ Paper method: Interpretation Table (integrated) evaluation
    gold_int = integrate_tables(gp, gc)
    pred_int = integrate_tables(pp, pc)
    int_pos_f1, int_none_acc = evaluate_tables(gold_int, pred_int, 'integrated')
    
    return {
        # Paper-reported metric (Interpretation Table)
        'step4_int_pos_f1': int_pos_f1,
        'step4_int_none_acc': int_none_acc,
        # For individual reference
        'pref_pos_f1': pref_pos_f1, 'pref_none_acc': pref_none_acc,
        'cons_pos_f1': cons_pos_f1, 'cons_none_acc': cons_none_acc,
    }

print("✅ Evaluation functions defined (matching paper Eq.1~6)")

✅ 평가 함수 정의 완료 (논문 Eq.1~6 일치)


## Data loading functions

In [4]:
def load_gold(log_name):
    """Load the gold-standard answers."""
    gold_path = os.path.join(GOLD_DIR, log_name)
    gold = {}
    for key, fname in [('step1_1', 'step1_1_gold.json'), ('step1_2', 'step1_2_gold.json'),
                       ('step2', 'step2_gold.json'), ('step3', 'step3_gold.json'),
                       ('step4', 'step4_gold.json')]:
        fpath = os.path.join(gold_path, fname)
        if os.path.exists(fpath):
            with open(fpath, 'r', encoding='utf-8') as f:
                gold[key] = json.load(f)
    return gold


def get_scope(fourstep_dir, log_name):
    """Extract scope (participant/restaurant range) from the 4-step final_results."""
    final_path = os.path.join(fourstep_dir, log_name, f'{log_name}_final_results.json')
    if not os.path.exists(final_path):
        return None, None
    with open(final_path, 'r', encoding='utf-8') as f:
        final = json.load(f)
    return set(final['step1']['participants']), set(final['step1']['restaurant_brands'])


def load_4step_predictions(fourstep_dir, log_name, technique='CoT'):
    """Load per-step prediction results of the 4-step pipeline (5 iterations)."""
    log_dir = os.path.join(fourstep_dir, log_name)
    results = []
    for it in ITERATIONS:
        pred = {}
        # Step 3
        s3_path = os.path.join(log_dir, f'{log_name}_Step3_{technique}_{it}.json')
        if os.path.exists(s3_path):
            with open(s3_path, 'r', encoding='utf-8') as f:
                s3 = json.load(f)
            pred['sentiment_table'] = s3['Result'].get('sentiment_table', [])
        # Step 4
        s4_path = os.path.join(log_dir, f'{log_name}_Step4_{technique}_{it}.json')
        if os.path.exists(s4_path):
            with open(s4_path, 'r', encoding='utf-8') as f:
                s4 = json.load(f)
            pred['preference_table'] = s4['Result'].get('preference_table', [])
            pred['constraint_table'] = s4['Result'].get('constraint_table', [])
        # Steps 1 and 2 are already covered by the technique_summary CSV
        if pred:
            results.append(pred)
    return results


def load_single_predictions(single_dir, log_name):
    """Load the single-prompt normalized results (5 iterations)."""
    log_dir = os.path.join(single_dir, log_name)
    results = []
    for it in ITERATIONS:
        norm_path = os.path.join(log_dir, f'{log_name}_normalized_iter{it}.json')
        if os.path.exists(norm_path):
            with open(norm_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            results.append(data['result'])
    return results


def get_all_logs(fourstep_dir):
    """List of evaluable conversations."""
    logs = []
    for d in sorted(os.listdir(fourstep_dir)):
        if d.endswith('_log') and os.path.isdir(os.path.join(fourstep_dir, d)):
            logs.append(d)
    return logs

print("✅ Data loading functions defined")

✅ 데이터 로드 함수 정의 완료


## Run the full evaluation

In [5]:
all_results = []  # results for all model x method x conversation x iteration

for model_label in FOURSTEP_DIRS:
    fourstep_dir = FOURSTEP_DIRS[model_label]
    single_dir = SINGLE_DIRS[model_label]
    
    logs = get_all_logs(fourstep_dir)
    print(f"\n{'='*60}")
    print(f"  {model_label}: {len(logs)} conversations")
    print(f"{'='*60}")
    
    for log_name in logs:
        gold = load_gold(log_name)
        if not gold:
            continue
        
        # Scope from 4-step
        scope_p, scope_r = get_scope(fourstep_dir, log_name)
        if scope_p is None:
            continue
        
        # Gold tables
        gold_s3 = gold.get('step3', {})
        gold_s3_table = gold_s3.get('perception_table', gold_s3.get('sentiment_table', []))
        gold_s4 = gold.get('step4', {})
        gold_pref = gold_s4.get('preference_table', [])
        gold_cons = gold_s4.get('constraint_table', [])
        
        # ── 4-step CoT ──
        fourstep_preds = load_4step_predictions(fourstep_dir, log_name)
        for it_idx, pred in enumerate(fourstep_preds):
            row = {'model': model_label, 'method': '4-step', 'log': log_name, 'iteration': it_idx + 1}
            
            # Step 3
            if 'sentiment_table' in pred:
                s3_micro = eval_step3_micro(gold_s3_table, pred['sentiment_table'], scope_p, scope_r)
                s3_macro = eval_step3_macro(gold_s3_table, pred['sentiment_table'])
                row.update(s3_micro)
                row.update(s3_macro)
            
            # Step 4 — paper method (integrated Interpretation Table)
            if 'preference_table' in pred:
                s4 = eval_step4_paper(gold_pref, gold_cons,
                                      pred['preference_table'], pred['constraint_table'],
                                      scope_p, scope_r)
                row.update(s4)
            
            all_results.append(row)
        
        # ── Single prompt ──
        single_preds = load_single_predictions(single_dir, log_name)
        for it_idx, pred in enumerate(single_preds):
            row = {'model': model_label, 'method': 'Single', 'log': log_name, 'iteration': it_idx + 1}
            
            # Step 3
            if 'sentiment_table' in pred:
                s3_micro = eval_step3_micro(gold_s3_table, pred['sentiment_table'], scope_p, scope_r)
                s3_macro = eval_step3_macro(gold_s3_table, pred['sentiment_table'])
                row.update(s3_micro)
                row.update(s3_macro)
            
            # Step 4 — paper method (integrated Interpretation Table)
            if 'preference_table' in pred:
                s4 = eval_step4_paper(gold_pref, gold_cons,
                                      pred['preference_table'], pred['constraint_table'],
                                      scope_p, scope_r)
                row.update(s4)
            
            all_results.append(row)
    
    print(f"  ✅ {model_label} done")

df = pd.DataFrame(all_results)
print(f"\nTotal {len(df)} rows")
df.head()


  GPT-5: 47 conversations
  ✅ GPT-5 완료

  GPT-4o: 47 conversations
  ✅ GPT-4o 완료


총 1400 rows


,model,method,log,iteration,F1_perception_micro,F1_perception_macro,step4_int_pos_f1,step4_int_none_acc,pref_pos_f1,pref_none_acc,cons_pos_f1,cons_none_acc
0,GPT-5,4-step,10_log,1,0.800000,0.741071,0.416667,0.375,0.444444,0.666667,0.533333,0.692308
1,GPT-5,4-step,10_log,2,0.758621,0.623543,0.496667,0.500,0.500000,0.500000,0.533333,0.846154
2,GPT-5,4-step,10_log,3,0.800000,0.683983,0.383333,0.250,0.444444,0.583333,0.466667,0.615385
3,GPT-5,4-step,10_log,4,0.758621,0.582792,0.450000,0.250,0.444444,0.666667,0.533333,0.615385
4,GPT-5,4-step,10_log,5,0.758621,0.623543,0.516667,0.375,0.388889,0.500000,0.533333,0.769231


## Results summary: comparison under the unified evaluation criteria

In [6]:
# Per-conversation average (iteration mean) -> per-model average
metrics_cols = ['F1_perception_micro', 'F1_perception_macro',
                'step4_int_pos_f1', 'step4_int_none_acc',
                'pref_pos_f1', 'cons_pos_f1',
                'pref_none_acc', 'cons_none_acc']

# Step 1: conversation x iteration -> per-conversation average
conv_avg = df.groupby(['model', 'method', 'log'])[metrics_cols].mean().reset_index()

# Print a clean table
print("=" * 100)
print("4-Step CoT vs Single Prompt — comparison under the paper's evaluation criteria")
print("=" * 100)

display_metrics = [
    ('F1_perception_micro', 'Step3 F1_PerceptionTable (Eq.4)'),
    ('F1_perception_macro', 'Step3 Macro F1 (ref)'),
    ('step4_int_pos_f1',   'Step4 Pos-F1_Integrated (Eq.5)'),
    ('step4_int_none_acc', 'Step4 None Acc_Integrated'),
    ('pref_pos_f1',        '  └ Preference Pos F1 (ref)'),
    ('cons_pos_f1',        '  └ Constraint Pos F1 (ref)'),
]

for model in ['GPT-5', 'GPT-4o']:
    print(f"\n{'─'*80}")
    print(f"  {model}")
    print(f"{'─'*80}")
    print(f"  {'Metric':<40} | {'4-Step CoT':>18} | {'Single':>18} | {'Δ':>10}")
    print(f"  {'-'*90}")
    
    for col, label in display_metrics:
        four_mask = (conv_avg['model'] == model) & (conv_avg['method'] == '4-step')
        single_mask = (conv_avg['model'] == model) & (conv_avg['method'] == 'Single')
        
        four_vals = conv_avg.loc[four_mask, col].dropna()
        single_vals = conv_avg.loc[single_mask, col].dropna()
        
        if len(four_vals) > 0 and len(single_vals) > 0:
            fm, fs = four_vals.mean(), four_vals.std()
            sm, ss = single_vals.mean(), single_vals.std()
            delta = sm - fm
            print(f"  {label:<40} | {fm:.4f} (±{fs:.4f}) | {sm:.4f} (±{ss:.4f}) | {delta:>+.4f}")

4-Step CoT vs Single Prompt — 논문 동일 평가 기준 비교

────────────────────────────────────────────────────────────────────────────────
  GPT-5
────────────────────────────────────────────────────────────────────────────────
  Metric                                   |         4-Step CoT |             Single |          Δ
  ------------------------------------------------------------------------------------------
  Step3 F1_PerceptionTable (Eq.4)          | 0.8426 (±0.0956) | 0.8422 (±0.0993) | -0.0003
  Step3 Macro F1 (참고)                      | 0.4632 (±0.1157) | 0.4555 (±0.1194) | -0.0077
  Step4 Pos-F1_Integrated (Eq.5)           | 0.3699 (±0.2068) | 0.3764 (±0.2059) | +0.0065
  Step4 None Acc_Integrated                | 0.5293 (±0.1866) | 0.5975 (±0.1943) | +0.0683
    └ Preference Pos F1 (참고)               | 0.3283 (±0.2324) | 0.3247 (±0.2325) | -0.0036
    └ Constraint Pos F1 (참고)               | 0.4675 (±0.3932) | 0.4615 (±0.3876) | -0.0060

──────────────────────────────────────────────

## Per-conversation detailed comparison (Paired)

In [7]:
# Paired comparison of 4-step vs Single per conversation
print("=" * 100)
print("Paired comparison: Single - 4-Step (positive = Single is better)")
print("=" * 100)

for model in ['GPT-5', 'GPT-4o']:
    print(f"\n{'─'*60}")
    print(f"  {model}")
    print(f"{'─'*60}")
    
    four = conv_avg[(conv_avg['model'] == model) & (conv_avg['method'] == '4-step')].set_index('log')
    single = conv_avg[(conv_avg['model'] == model) & (conv_avg['method'] == 'Single')].set_index('log')
    
    common_logs = sorted(set(four.index) & set(single.index))
    
    paired_metrics = [
        ('F1_perception_micro', 'Step3 F1_PerceptionTable'),
        ('step4_int_pos_f1',   'Step4 Pos-F1_Integrated'),
    ]
    
    for col, label in paired_metrics:
        diffs = []
        for log in common_logs:
            if col in four.columns and col in single.columns:
                d = single.loc[log, col] - four.loc[log, col]
                if not np.isnan(d):
                    diffs.append(d)
        
        if diffs:
            diffs = np.array(diffs)
            n_better = np.sum(diffs > 0.01)
            n_same = np.sum(np.abs(diffs) <= 0.01)
            n_worse = np.sum(diffs < -0.01)
            print(f"  {label:<30}: mean Δ={np.mean(diffs):>+.4f}  "
                  f"(Single↑:{n_better}  same:{n_same}  4-step↑:{n_worse}  /  {len(diffs)} convs)")

Paired 비교: Single - 4-Step (양수 = Single이 더 좋음)

────────────────────────────────────────────────────────────
  GPT-5
────────────────────────────────────────────────────────────
  Step3 F1_PerceptionTable      : mean Δ=-0.0003  (Single↑:15  동일:13  4-step↑:19  /  47 convs)
  Step4 Pos-F1_Integrated       : mean Δ=+0.0065  (Single↑:23  동일:10  4-step↑:14  /  47 convs)

────────────────────────────────────────────────────────────
  GPT-4o
────────────────────────────────────────────────────────────
  Step3 F1_PerceptionTable      : mean Δ=+0.0058  (Single↑:17  동일:9  4-step↑:20  /  46 convs)
  Step4 Pos-F1_Integrated       : mean Δ=-0.0801  (Single↑:5  동일:13  4-step↑:28  /  46 convs)

────────────────────────────────────────────────────────────
────────────────────────────────────────────────────────────
  Step3 F1_PerceptionTable      : mean Δ=-0.0248  (Single↑:16  동일:2  4-step↑:28  /  46 convs)
  Step4 Pos-F1_Integrated       : mean Δ=-0.1151  (Single↑:5  동일:5  4-step↑:36  /  46 convs)


## Save CSV

In [8]:
# Full detailed results
df.to_csv('appendix/results/comparison/single_vs_4step_appendix/results/comparison/unified_eval_detail.csv', index=False)

# Per-conversation average results
conv_avg.to_csv('appendix/results/comparison/single_vs_4step_appendix/results/comparison/unified_eval_by_conv.csv', index=False)

# Per-model summary
model_summary = conv_avg.groupby(['model', 'method'])[metrics_cols].agg(['mean', 'std']).reset_index()
model_summary.to_csv('appendix/results/comparison/single_vs_4step_appendix/results/comparison/unified_eval_summary.csv', index=False)

print("✅ Saved:")
print("  - appendix/results/comparison/single_vs_4step_appendix/results/comparison/unified_eval_detail.csv (all iterations)")
print("  - appendix/results/comparison/single_vs_4step_appendix/results/comparison/unified_eval_by_conv.csv (per-conversation average)")
print("  - appendix/results/comparison/single_vs_4step_appendix/results/comparison/unified_eval_summary.csv (per-model summary)")
print()
print("📌 Paper-aligned metrics:")
print("  - Step 3: F1_perception_micro = F1_PerceptionTable (Eq.4)")
print("  - Step 4: step4_int_pos_f1 = Positive-F1_InterpretationTable (Eq.5)")

✅ 저장 완료:
  - single_vs_4step_unified_eval_detail.csv (전체 iteration별)
  - single_vs_4step_unified_eval_by_conv.csv (대화별 평균)
  - single_vs_4step_unified_eval_summary.csv (모델별 요약)

📌 논문 일치 지표:
  - Step 3: F1_perception_micro = F1_PerceptionTable (Eq.4)
  - Step 4: step4_int_pos_f1 = Positive-F1_InterpretationTable (Eq.5)
